In [ ]:
# Checkpoint 3: Extended DistilBERT Reimplementation for IMDB Sentiment Analysis

# ECE570 - Spring 2025
# Description: This notebook refines and extends the preliminary DistilBERT implementation.
#              It includes advanced hyperparameter tuning, regularization, sequence length variation,
#              and experimental generative scoring integration based on prior project proposal.

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ========================
# Environment Setup
# ========================
!pip install --upgrade transformers datasets
!pip install transformers datasets scikit-learn -q

import torch
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DistilBertConfig
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.50.3
    Uninstalling transformers-4.50.3:
      Successfully uninstalled transformers-4.50.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
to

In [3]:
# ========================
# Data Loading and Tokenization
# ========================

raw_datasets = load_dataset("imdb")
raw_train = raw_datasets["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = raw_train["train"]
val_dataset = raw_train["test"]
test_dataset = raw_datasets["test"]

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples, max_len):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=max_len)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [4]:
# ========================
# Compute Metrics
# ========================

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

In [ ]:
# ========================
# First Round Hyperparameter Grid Search + Time-Accuracy Tradeoff
# ========================

hyper_grid = {
    "learning_rate": [2e-5, 3e-5],
    "batch_size": [16, 32],
    "epochs": [2, 3, 5]
}

results = []

for lr in hyper_grid["learning_rate"]:
    for bs in hyper_grid["batch_size"]:
        for ep in hyper_grid["epochs"]:

            tokenized_train = train_dataset.map(lambda x: tokenize_function(x, 128), batched=True)
            tokenized_val = val_dataset.map(lambda x: tokenize_function(x, 128), batched=True)

            tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
            tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

            model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)

            args = TrainingArguments(
                output_dir="./results",
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                num_train_epochs=ep,
                eval_strategy="epoch",
                save_strategy="epoch",
                logging_dir="./logs",
                logging_steps=500,
                logging_strategy = "steps",
                load_best_model_at_end=True
            )

            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=tokenized_train,
                eval_dataset=tokenized_val,
                compute_metrics=compute_metrics,
                tokenizer=tokenizer
            )

            start_time = time.time()
            trainer.train()
            end_time = time.time()
            train_duration = end_time - start_time

            eval_result = trainer.evaluate()
            results.append({
                "lr": lr,
                "bs": bs,
                "epochs": ep,
                "accuracy": eval_result['eval_accuracy'],
                "train_time": train_duration
            })

# Analyze trade-off between accuracy and training time
results_df = pd.DataFrame(results)
results_df["accuracy_per_min"] = results_df["accuracy"] / (results_df["train_time"] / 60)

best_tradeoff = results_df.loc[results_df["accuracy_per_min"].idxmax()]
print("Best tradeoff configuration:")
print(best_tradeoff)



Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-5-79bb383f8bac>:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:


Abort: 

 ··········


In [ ]:
# ========================
# Round 1.5: Sequence Length Comparison
# ========================
best_params = best_tradeoff.to_dict()
max_lengths = [64, 256, 384, 512] #Skipping 128 as that was the length used in grid search
length_results = []

for max_len in max_lengths:
    tokenized_train = train_dataset.map(lambda x: tokenize_function(x, max_len), batched=True)
    tokenized_val = val_dataset.map(lambda x: tokenize_function(x, max_len), batched=True)

    tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)

    args = TrainingArguments(
        output_dir=f"./results-{max_len}",
        learning_rate=best_params['lr'],
        per_device_train_batch_size=int(best_params['bs']),
        num_train_epochs=best_params['epochs'],
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        logging_steps=500,
        logging_strategy = "steps",
        load_best_model_at_end=True
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )
    start_time = time.time()
    trainer.train()
    end_time = time.time()
    train_duration = end_time - start_time
    eval_result = trainer.evaluate()
    length_results.append({"max_length": max_len, "accuracy": eval_result["eval_accuracy"], "train_duration": train_duration})
    df_length = pd.DataFrame(length_results)
    df_length["accuracy_per_min"] = df_length["accuracy"] / (df_length["train_duration"] / 60)

    # Identify the best tradeoff based on accuracy per minute
    best_tradeoff_length = df_length.loc[df_length["accuracy_per_min"].idxmax()]


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-31-d44af8af532a>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.464700,0.381239,0.823600,0.812068,0.843176,0.827330
2,0.319300,0.387886,0.827000,0.830980,0.822027,0.826479


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-31-d44af8af532a>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.320600,0.231712,0.904800,0.892498,0.920990,0.906520
2,0.185400,0.248195,0.907200,0.912692,0.901038,0.906827


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.296100,0.217951,0.913600,0.937184,0.887071,0.911439
2,0.161000,0.236151,0.918400,0.916270,0.921389,0.918822


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-31-d44af8af532a>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.285800,0.195171,0.922400,0.926329,0.918196,0.922244
2,0.160900,0.220258,0.923200,0.921366,0.925778,0.923567


In [ ]:
# ========================
# Round 2
# ========================
hyper_grid = {
    "learning_rate": [4e-5, 8e-5],
    "batch_size": [48, 64],
    "max_length": [64, 128]
}

results = []

for lr in hyper_grid["learning_rate"]:
    for bs in hyper_grid["batch_size"]:
        for len in hyper_grid["max_length"]:

            tokenized_train = train_dataset.map(lambda x: tokenize_function(x, len), batched=True)
            tokenized_val = val_dataset.map(lambda x: tokenize_function(x, len), batched=True)

            tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
            tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

            model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)

            args = TrainingArguments(
                output_dir="./results",
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                num_train_epochs=2,
                eval_strategy="epoch",
                save_strategy="epoch",
                logging_dir="./logs",
                logging_steps=500,
                logging_strategy = "steps",
                load_best_model_at_end=True
            )

            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=tokenized_train,
                eval_dataset=tokenized_val,
                compute_metrics=compute_metrics,
                tokenizer=tokenizer
            )

            start_time = time.time()
            trainer.train()
            end_time = time.time()
            train_duration = end_time - start_time

            eval_result = trainer.evaluate()
            results.append({
                "lr": lr,
                "bs": bs,
                "epochs": ep,
                "accuracy": eval_result['eval_accuracy'],
                "train_time": train_duration
            })

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-9-e36a2039a897>:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.377710,0.824000,0.817083,0.835994,0.826430
2,0.424500,0.386606,0.829200,0.835500,0.820830,0.828100


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.302425,0.868000,0.846732,0.899441,0.872291
2,0.341400,0.320946,0.868800,0.870593,0.867119,0.868852


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-9-e36a2039a897>:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.382885,0.820600,0.814873,0.830806,0.822762
2,0.380400,0.391554,0.832800,0.833733,0.832402,0.833067


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-9-e36a2039a897>:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.298782,0.871400,0.878813,0.862330,0.870493
2,0.300600,0.314004,0.872600,0.874549,0.870710,0.872625


In [ ]:
# ========================
# Round 3
# ========================
#Confirmation that max_length is independent from other variables so reducing to 64 for faster training
#Reintroducing higher epochs to test for potential benefits
hyper_grid = {
    "learning_rate": [8e-5, 12e-5, 24e-5],
    "batch_size": [64, 128],
    "epoch": [2,5,7]
}

results = []

for lr in hyper_grid["learning_rate"]:
    for bs in hyper_grid["batch_size"]:
        for ep in hyper_grid["epoch"]:

            tokenized_train = train_dataset.map(lambda x: tokenize_function(x, 64), batched=True)
            tokenized_val = val_dataset.map(lambda x: tokenize_function(x, 64), batched=True)

            tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
            tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

            model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)

            args = TrainingArguments(
                output_dir="./results",
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                num_train_epochs=ep,
                eval_strategy="epoch",
                save_strategy="epoch",
                logging_dir="./logs",
                logging_steps=500,
                logging_strategy = "steps",
                load_best_model_at_end=True
            )

            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=tokenized_train,
                eval_dataset=tokenized_val,
                compute_metrics=compute_metrics,
                tokenizer=tokenizer
            )

            start_time = time.time()
            trainer.train()
            end_time = time.time()
            train_duration = end_time - start_time

            eval_result = trainer.evaluate()
            results.append({
                "lr": lr,
                "bs": bs,
                "epochs": ep,
                "accuracy": eval_result['eval_accuracy'],
                "train_time": train_duration
            })

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-4-cb70c80ce6c4>:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.407640,0.809600,0.816626,0.799681,0.808065
2,No log,0.411725,0.822600,0.822896,0.823224,0.823060


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-4-cb70c80ce6c4>:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.415365,0.806800,0.853860,0.741421,0.793678
2,No log,0.468260,0.805600,0.756008,0.903831,0.823337
3,No log,0.586645,0.821200,0.849220,0.782123,0.814292
4,0.281000,0.701309,0.818600,0.810968,0.832003,0.821351
5,0.281000,0.910345,0.822600,0.813881,0.837590,0.825565


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-4-cb70c80ce6c4>:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.440593,0.791800,0.836163,0.727055,0.777801
2,No log,0.436972,0.805200,0.775540,0.860335,0.815740
3,No log,0.580945,0.796600,0.841044,0.732642,0.783109
4,0.314700,0.722997,0.803000,0.798352,0.812051,0.805143
5,0.314700,0.887853,0.797600,0.808168,0.781724,0.794726
6,0.314700,1.066996,0.796800,0.800403,0.792099,0.796229
7,0.032100,1.116652,0.797600,0.801940,0.791700,0.796787


In [2]:
# ============================
# Loss Callback
# ============================
class PlotMetricsCallback:
    def __init__(self):
        self.train_loss = []
        self.eval_accuracy = []
        self.steps = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs:
            self.train_loss.append(logs["loss"])
        if "eval_accuracy" in logs:
            self.eval_accuracy.append(logs["eval_accuracy"])
        self.steps.append(state.global_step)

def plot_trainer_curves(callback):
    plt.plot(callback.steps, callback.train_loss, label="Train Loss")
    plt.plot(callback.steps, callback.eval_accuracy, label="Eval Accuracy")
    plt.xlabel("Steps")
    plt.ylabel("Metric")
    plt.title("Training Loss & Eval Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("fig_training_curves.png")
    plt.close()

In [ ]:
# ========================
# Optimized Model Training 
# ========================
tokenized_train = train_dataset.map(lambda x: tokenize_function(x, 256), batched=True)
tokenized_val = val_dataset.map(lambda x: tokenize_function(x, 256), batched=True)

tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)

args = TrainingArguments(
    output_dir="./results",
    learning_rate=12e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=500,
    logging_strategy = "steps",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

start_time = time.time()
trainer.train()
end_time = time.time()
train_duration = end_time - start_time

eval_result = trainer.evaluate()


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-11-726c22ab9b27>:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.253356,0.892000,0.925910,0.852753,0.887827
2,0.250500,0.248968,0.910200,0.909272,0.911812,0.910540


In [13]:
results = []
results.append({
    "lr": 12e-5,
    "bs": 64,
    "epochs": 2,
    "accuracy": eval_result['eval_accuracy'],
    "train_time": train_duration
})
print("Final Results:")
print(results)

Final Results:
[{'lr': 0.00012, 'bs': 64, 'epochs': 2, 'accuracy': 0.9102, 'train_time': 897.2727465629578}]


In [15]:
#Testing Final Model
test_dataset = test_dataset.map(lambda x: tokenize_function(x, 256), batched=True)
predictions = trainer.predict(test_dataset)

# Get predicted labels
predicted_labels = np.argmax(predictions.predictions, axis=1)

# Calculate test accuracy
test_accuracy = accuracy_score(predictions.label_ids, predicted_labels)

print(f"Test Accuracy: {test_accuracy}")
# Access validation accuracy from the trainer's evaluation results
validation_accuracy = trainer.evaluate()['eval_accuracy']

print(f"Validation Accuracy: {validation_accuracy}")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Test Accuracy: 0.90776


Validation Accuracy: 0.9102
